# FluxPipeline - GIF Creation

This notebook demonstrates how to create animated GIFs from generated images.

## What You'll Learn
- Generate image sequences
- Create smooth transitions
- Export as animated GIF
- Control animation parameters

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import torch
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import Image as IPImage
from datetime import datetime

from pipeline import FluxPipeline
from core import SeedProfile
from config import setup_environment, logger
from utils import setup_workspace

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Initialize Pipeline

In [ ]:
setup_environment()
workspace = setup_workspace()

pipeline = FluxPipeline(workspace=workspace)
pipeline.load_model()
print("✅ Pipeline ready!")

## Method 1: Simple Sequence GIF

Generate a sequence of images with similar prompts.

In [ ]:
# Create a sequence of related prompts
sequence_prompts = [
    "A landscape at dawn with soft pink sky",
    "A landscape at morning with golden sunlight",
    "A landscape at noon with bright blue sky",
    "A landscape at sunset with orange glow",
    "A landscape at dusk with purple twilight",
    "A landscape at night with stars",
]

print(f"Generating {len(sequence_prompts)} frames...\n")

frames = []

for idx, prompt in enumerate(sequence_prompts, 1):
    print(f"[{idx}/{len(sequence_prompts)}] {prompt}")
    
    image, seed = pipeline.generate_image(
        prompt=prompt,
        num_inference_steps=4,
        height=512,
        width=512,
        seed_profile=SeedProfile.BALANCED
    )
    
    if image:
        frames.append(image)
        print(f"  ✅ Frame {idx} generated (seed: {seed})")

print(f"\n✅ Generated {len(frames)} frames!")

In [ ]:
# Preview frames
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, frame in enumerate(frames):
    axes[idx].imshow(frame)
    axes[idx].axis('off')
    axes[idx].set_title(f"Frame {idx + 1}", fontsize=10)

plt.tight_layout()
plt.show()

## Create and Save GIF

In [ ]:
# Save as GIF
gif_path = workspace / f"day_cycle_{datetime.now().strftime('%Y%m%d_%H%M%S')}.gif"

frames[0].save(
    gif_path,
    save_all=True,
    append_images=frames[1:],
    duration=800,  # milliseconds per frame
    loop=0  # 0 = infinite loop
)

print(f"✅ GIF saved to: {gif_path}")
print(f"   - Frames: {len(frames)}")
print(f"   - Duration per frame: 800ms")
print(f"   - Total duration: {len(frames) * 0.8:.1f}s")

In [ ]:
# Display the GIF in notebook
IPImage(filename=str(gif_path))

## Method 2: Transformation GIF

Create a transformation sequence (e.g., season change).

In [ ]:
# Seasonal transformation
season_prompts = [
    "A beautiful tree in spring with pink blossoms",
    "A beautiful tree in summer with lush green leaves",
    "A beautiful tree in autumn with orange and red leaves",
    "A beautiful tree in winter covered in snow",
]

print("Generating seasonal transformation...\n")

season_frames = []

for idx, prompt in enumerate(season_prompts, 1):
    print(f"[{idx}/{len(season_prompts)}] {prompt}")
    
    image, seed = pipeline.generate_image(
        prompt=prompt,
        num_inference_steps=4,
        height=768,
        width=768,
        seed=42  # Use same seed for consistency
    )
    
    if image:
        season_frames.append(image)
        print(f"  ✅ Generated")

# Create GIF with slower transitions
season_gif_path = workspace / f"seasons_{datetime.now().strftime('%Y%m%d_%H%M%S')}.gif"

season_frames[0].save(
    season_gif_path,
    save_all=True,
    append_images=season_frames[1:],
    duration=1500,  # 1.5 seconds per frame
    loop=0
)

print(f"\n✅ Seasonal GIF saved to: {season_gif_path}")

In [ ]:
# Display seasonal GIF
IPImage(filename=str(season_gif_path))

## Method 3: Using Existing Script

FluxPipeline includes a `generate_transformation_gif.py` script.

In [ ]:
# You can also use the built-in script from command line:
# python generate_transformation_gif.py \
#     --start-prompt "A landscape in summer" \
#     --end-prompt "A landscape in winter" \
#     --steps 6 \
#     --duration 1000

print("To use the built-in GIF generator, run from terminal:")
print("")
print("python generate_transformation_gif.py \\")
print("    --start-prompt 'Your starting scene' \\")
print("    --end-prompt 'Your ending scene' \\")
print("    --steps 8 \\")
print("    --duration 800")

## Advanced: Custom GIF Parameters

In [ ]:
def create_custom_gif(frames, output_path, **kwargs):
    """Create GIF with custom parameters.
    
    Args:
        frames: List of PIL Images
        output_path: Path to save GIF
        **kwargs: Additional PIL save parameters
            - duration: milliseconds per frame (default: 500)
            - loop: number of loops, 0=infinite (default: 0)
            - optimize: optimize palette (default: True)
            - quality: JPEG quality for optimization (default: 85)
    """
    params = {
        'save_all': True,
        'append_images': frames[1:],
        'duration': kwargs.get('duration', 500),
        'loop': kwargs.get('loop', 0),
        'optimize': kwargs.get('optimize', True),
    }
    
    frames[0].save(output_path, **params)
    
    file_size = output_path.stat().st_size / 1024 / 1024  # MB
    print(f"✅ GIF created:")
    print(f"   - Path: {output_path}")
    print(f"   - Frames: {len(frames)}")
    print(f"   - Size: {file_size:.2f} MB")
    print(f"   - Duration: {params['duration']}ms/frame")

# Example: Create optimized GIF
optimized_path = workspace / "optimized_animation.gif"
create_custom_gif(
    frames,
    optimized_path,
    duration=600,
    loop=0,
    optimize=True
)

## Tips for Better GIFs

1. **Prompt Consistency**: Use similar scene structure across prompts
2. **Fixed Seed**: Consider using the same seed for smoother transitions
3. **Frame Count**: 6-12 frames work well for most animations
4. **Duration**: 500-1000ms per frame is usually good
5. **Resolution**: Lower resolution = smaller file size
6. **Optimization**: Enable optimize=True to reduce file size

## Next Steps

- **[04_custom_prompts.ipynb](04_custom_prompts.ipynb)** - Improve your prompts
- **[05_memory_optimization.ipynb](05_memory_optimization.ipynb)** - Handle longer sequences
- **[06_seed_management.ipynb](06_seed_management.ipynb)** - Control randomness

## Cleanup

In [ ]:
import gc

del pipeline
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ Cleanup complete!")